## Go through all settings for minimum_stop_duration_mins and spatial_radius_km

Duration: 5 minutes, 15 minutes, 20 minutes, 30 minutes
Radius: 0.1km, 0.2km, 0.3km, 0.5km

Findings:
* 5 minutes is definitely too low, splits up trajectories. Even if radius is lower (100km), this resolves Fed Way (slow movement) but not Harborview (gap in sampling data)
* No_data_for_minutes (tested 5, 30, 60) ends up combining trajectories in a way that is undesired -- we may be able to manually split these, but it's effectively same as counting these gaps as stops. Key issue may be deciding on the proper thresholds for time/distance of stop
* 200m seems to help more than 100m (some stops slightly more than 100m together) but 300m and 500m don't seem to help, besides capturing a bit of trajectory movement points within the stop.
* 15 versus 20 minutes seems like a tradeoff, where 20 allows to connect trips along a journey better, but hard to tell if it's ignoring actual momentary stops somewhere (prob some but not others). 30 minutes is very similar to 20 minutes, might be more appropriate due to Spectus' sparsity but definitely overlooks some human behavior.

Suggestions to fix issues:
* Also applying no_data_for_minutes to trajectory-calculation side, though we will likely have unforeseen functionality from trajectories being improperly merged.
* Setting a minimum number of datapoints to count as a stay (filtering when calculating trajectories after clustering)

In [ ]:
# Generate Spectus database metrics for 2000 users
from datetime import date
import pandas as pd
from pathlib import Path
from mode_analysis_main.src.calculate_speed import haversine
from mode_analysis_main.src.plot import plot_trip_on_map

from collections import namedtuple
Trajectory = namedtuple('Trajectory', ['id', 'source', 'df'])
Cluster = namedtuple('Cluster', ['id', 'source', 'df'])

today = date.today().strftime("%Y%m%d")

input_path = f'./outputs/seattle_2000_smaller_5min_100m_no_clustering.csv'

input_df = pd.read_csv(input_path, low_memory=False)
input_df = input_df.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
input_df['timestamp'] = pd.to_datetime(input_df['timestamp'])

processing_limit = 40
user_count = 0
total_trips_count = 0
filtered_trips_count = 0
for user, user_df in input_df.groupby('user_ID'):

    if total_trips_count >= processing_limit:
        break

    count = 0
    try:
        print(f'Processing user {user_count}')

        # Remove stop points
        trajectories_df = user_df[user_df['traj_id'] != -99]
        trajectory_dfs = [Trajectory(traj_id, 'Trajectory', df) for traj_id, df in trajectories_df.groupby('traj_id')]

        clusters_df = user_df[user_df['cluster'] != -99]
        cluster_dfs = [Cluster(cluster, 'Cluster', df) for cluster, df in clusters_df.groupby('cluster')]

        all_dfs = trajectory_dfs + cluster_dfs
        all_dfs_ordered = sorted(all_dfs, key=lambda tuple: tuple.df.iloc[0]['timestamp'])

        # trajectories_df['traj_id'] = trajectories_df['user_ID'].astype(str) + '_' + trajectories_df['traj_id'].astype(str)

        for tuple in all_dfs_ordered:
            if count >= processing_limit:
                break

            points_dataframe = tuple.df

            print(f'Processing {tuple.source} {tuple.id}')
            print(f'Starting point: ({points_dataframe.iloc[0]["lat"]}, {points_dataframe.iloc[0]["lng"]})')
            print(f'Ending point: ({points_dataframe.iloc[-1]["lat"]}, {points_dataframe.iloc[-1]["lng"]})')
            print(f'Number of points: {len(points_dataframe)}')

            distance_beginning_to_end = haversine(points_dataframe.iloc[0]['lat'],
                                                  points_dataframe.iloc[0]['lng'],
                                                  points_dataframe.iloc[-1]['lat'],
                                                  points_dataframe.iloc[-1]['lng'])
            print(f'Distance covered (first to last): {distance_beginning_to_end}')

            points_dataframe_minus_last = points_dataframe[:-1]
            bounds = (points_dataframe_minus_last['lng'].min(),
                      points_dataframe_minus_last['lat'].min(),
                      points_dataframe_minus_last['lng'].max(),
                      points_dataframe_minus_last['lat'].max())
            corner1 = (bounds[1], bounds[0])
            corner2 = (bounds[3], bounds[2])
            distance_across = haversine(corner1[0], corner1[1], corner2[0], corner2[1])
            print(f'Max distance covered (excluding last): {distance_across}')

            ## ADD TIME GAP TO DEMONSTRATE WHETHER LONG GAPS CREATES STOPS OF ONE DATAPOINT WHENEVER TIME + DISTANCE ARE LARGE ENOUGH

            if len(points_dataframe) < 2:
                longest_time_gap_minutes = 0.0
                final_time_gap = 0.0
            else:
                longest_time_gap_minutes = (points_dataframe['timestamp'] - points_dataframe['timestamp'].shift(1)).max().total_seconds() / 60
                final_time_gap = (points_dataframe.iloc[-1]['timestamp'] - points_dataframe.iloc[-2]['timestamp']).total_seconds() / 60
            print(f'Longest time gap: {longest_time_gap_minutes}')
            print(f'Time gap before end: {final_time_gap}')

            plot_trip_on_map(points_dataframe, include_transit=False)

            count += 1

    except Exception as e:
        print(f"Unexpected error processing user: {user}")
        print(f"Error: {e}")

    user_count += 1
    print(f'Appended {count} rows')
    total_trips_count += count

print("Done!")
print(f"Total trips processed: {total_trips_count}")
print(f"Trips filtered: {filtered_trips_count}")

In [ ]:
# Generate Spectus database metrics for 2000 users
from datetime import date
import pandas as pd
from pathlib import Path
from mode_analysis_main.src.calculate_speed import haversine
from mode_analysis_main.src.plot import plot_trip_on_map

from collections import namedtuple
Trajectory = namedtuple('Trajectory', ['id', 'source', 'df'])
Cluster = namedtuple('Cluster', ['id', 'source', 'df'])

today = date.today().strftime("%Y%m%d")

input_path = f'./outputs/seattle_2000_smaller_5min_200m_no_clustering.csv'

input_df = pd.read_csv(input_path, low_memory=False)
input_df = input_df.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
input_df['timestamp'] = pd.to_datetime(input_df['timestamp'])

processing_limit = 40
user_count = 0
total_trips_count = 0
filtered_trips_count = 0
for user, user_df in input_df.groupby('user_ID'):

    if total_trips_count >= processing_limit:
        break

    count = 0
    try:
        print(f'Processing user {user_count}')

        # Remove stop points
        trajectories_df = user_df[user_df['traj_id'] != -99]
        trajectory_dfs = [Trajectory(traj_id, 'Trajectory', df) for traj_id, df in trajectories_df.groupby('traj_id')]

        clusters_df = user_df[user_df['cluster'] != -99]
        cluster_dfs = [Cluster(cluster, 'Cluster', df) for cluster, df in clusters_df.groupby('cluster')]

        all_dfs = trajectory_dfs + cluster_dfs
        all_dfs_ordered = sorted(all_dfs, key=lambda tuple: tuple.df.iloc[0]['timestamp'])

        # trajectories_df['traj_id'] = trajectories_df['user_ID'].astype(str) + '_' + trajectories_df['traj_id'].astype(str)

        for tuple in all_dfs_ordered:
            if count >= processing_limit:
                break

            points_dataframe = tuple.df

            print(f'Processing {tuple.source} {tuple.id}')
            print(f'Starting point: ({points_dataframe.iloc[0]["lat"]}, {points_dataframe.iloc[0]["lng"]})')
            print(f'Ending point: ({points_dataframe.iloc[-1]["lat"]}, {points_dataframe.iloc[-1]["lng"]})')
            print(f'Number of points: {len(points_dataframe)}')

            distance_beginning_to_end = haversine(points_dataframe.iloc[0]['lat'],
                                                  points_dataframe.iloc[0]['lng'],
                                                  points_dataframe.iloc[-1]['lat'],
                                                  points_dataframe.iloc[-1]['lng'])
            print(f'Distance covered (first to last): {distance_beginning_to_end}')

            points_dataframe_minus_last = points_dataframe[:-1]
            bounds = (points_dataframe_minus_last['lng'].min(),
                      points_dataframe_minus_last['lat'].min(),
                      points_dataframe_minus_last['lng'].max(),
                      points_dataframe_minus_last['lat'].max())
            corner1 = (bounds[1], bounds[0])
            corner2 = (bounds[3], bounds[2])
            distance_across = haversine(corner1[0], corner1[1], corner2[0], corner2[1])
            print(f'Max distance covered (excluding last): {distance_across}')

            ## ADD TIME GAP TO DEMONSTRATE WHETHER LONG GAPS CREATES STOPS OF ONE DATAPOINT WHENEVER TIME + DISTANCE ARE LARGE ENOUGH

            if len(points_dataframe) < 2:
                longest_time_gap_minutes = 0.0
                final_time_gap = 0.0
            else:
                longest_time_gap_minutes = (points_dataframe['timestamp'] - points_dataframe['timestamp'].shift(1)).max().total_seconds() / 60
                final_time_gap = (points_dataframe.iloc[-1]['timestamp'] - points_dataframe.iloc[-2]['timestamp']).total_seconds() / 60
            print(f'Longest time gap: {longest_time_gap_minutes}')
            print(f'Time gap before end: {final_time_gap}')

            plot_trip_on_map(points_dataframe, include_transit=False)

            count += 1

    except Exception as e:
        print(f"Unexpected error processing user: {user}")
        print(f"Error: {e}")

    user_count += 1
    print(f'Appended {count} rows')
    total_trips_count += count

print("Done!")
print(f"Total trips processed: {total_trips_count}")
print(f"Trips filtered: {filtered_trips_count}")

In [ ]:
# Generate Spectus database metrics for 2000 users
from datetime import date
import pandas as pd
from pathlib import Path
from mode_analysis_main.src.calculate_speed import haversine
from mode_analysis_main.src.plot import plot_trip_on_map

from collections import namedtuple
Trajectory = namedtuple('Trajectory', ['id', 'source', 'df'])
Cluster = namedtuple('Cluster', ['id', 'source', 'df'])

today = date.today().strftime("%Y%m%d")

input_path = f'./outputs/seattle_2000_smaller_15min_200m_no_clustering.csv'

input_df = pd.read_csv(input_path, low_memory=False)
input_df = input_df.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
input_df['timestamp'] = pd.to_datetime(input_df['timestamp'])

processing_limit = 50
user_count = 0
total_trips_count = 0
filtered_trips_count = 0
for user, user_df in input_df.groupby('user_ID'):

    if total_trips_count >= processing_limit:
        break

    count = 0
    try:
        print(f'Processing user {user_count}')

        # Remove stop points
        trajectories_df = user_df[user_df['traj_id'] != -99]
        trajectory_dfs = [Trajectory(traj_id, 'Trajectory', df) for traj_id, df in trajectories_df.groupby('traj_id')]

        clusters_df = user_df[user_df['cluster'] != -99]
        cluster_dfs = [Cluster(cluster, 'Cluster', df) for cluster, df in clusters_df.groupby('cluster')]

        all_dfs = trajectory_dfs + cluster_dfs
        all_dfs_ordered = sorted(all_dfs, key=lambda tuple: tuple.df.iloc[0]['timestamp'])

        # trajectories_df['traj_id'] = trajectories_df['user_ID'].astype(str) + '_' + trajectories_df['traj_id'].astype(str)

        for tuple in all_dfs_ordered:
            if count >= processing_limit:
                break

            points_dataframe = tuple.df

            print(f'Processing {tuple.source} {tuple.id}')
            print(f'Starting point: ({points_dataframe.iloc[0]["lat"]}, {points_dataframe.iloc[0]["lng"]})')
            print(f'Ending point: ({points_dataframe.iloc[-1]["lat"]}, {points_dataframe.iloc[-1]["lng"]})')
            print(f'Number of points: {len(points_dataframe)}')

            distance_beginning_to_end = haversine(points_dataframe.iloc[0]['lat'],
                                                  points_dataframe.iloc[0]['lng'],
                                                  points_dataframe.iloc[-1]['lat'],
                                                  points_dataframe.iloc[-1]['lng'])
            print(f'Distance covered (first to last): {distance_beginning_to_end}')

            points_dataframe_minus_last = points_dataframe[:-1]
            bounds = (points_dataframe_minus_last['lng'].min(),
                      points_dataframe_minus_last['lat'].min(),
                      points_dataframe_minus_last['lng'].max(),
                      points_dataframe_minus_last['lat'].max())
            corner1 = (bounds[1], bounds[0])
            corner2 = (bounds[3], bounds[2])
            distance_across = haversine(corner1[0], corner1[1], corner2[0], corner2[1])
            print(f'Max distance covered (excluding last): {distance_across}')

            ## ADD TIME GAP TO DEMONSTRATE WHETHER LONG GAPS CREATES STOPS OF ONE DATAPOINT WHENEVER TIME + DISTANCE ARE LARGE ENOUGH

            if len(points_dataframe) < 2:
                longest_time_gap_minutes = 0.0
                final_time_gap = 0.0
            else:
                longest_time_gap_minutes = (points_dataframe['timestamp'] - points_dataframe['timestamp'].shift(1)).max().total_seconds() / 60
                final_time_gap = (points_dataframe.iloc[-1]['timestamp'] - points_dataframe.iloc[-2]['timestamp']).total_seconds() / 60
            print(f'Longest time gap: {longest_time_gap_minutes}')
            print(f'Time gap before end: {final_time_gap}')

            plot_trip_on_map(points_dataframe, include_transit=False)

            count += 1

    except Exception as e:
        print(f"Unexpected error processing user: {user}")
        print(f"Error: {e}")

    user_count += 1
    print(f'Appended {count} rows')
    total_trips_count += count

print("Done!")
print(f"Total trips processed: {total_trips_count}")
print(f"Trips filtered: {filtered_trips_count}")

In [ ]:
# Generate Spectus database metrics for 2000 users
from datetime import date
import pandas as pd
from pathlib import Path
from mode_analysis_main.src.calculate_speed import haversine
from mode_analysis_main.src.plot import plot_trip_on_map

from collections import namedtuple
Trajectory = namedtuple('Trajectory', ['id', 'source', 'df'])
Cluster = namedtuple('Cluster', ['id', 'source', 'df'])

today = date.today().strftime("%Y%m%d")

input_path = f'./outputs/seattle_2000_smaller_5min_200m_5minNoData_no_clustering.csv'

input_df = pd.read_csv(input_path, low_memory=False)
input_df = input_df.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
input_df['timestamp'] = pd.to_datetime(input_df['timestamp'])

processing_limit = 50
user_count = 0
total_trips_count = 0
filtered_trips_count = 0
for user, user_df in input_df.groupby('user_ID'):

    if total_trips_count >= processing_limit:
        break

    count = 0
    try:
        print(f'Processing user {user_count}')

        # Remove stop points
        trajectories_df = user_df[user_df['traj_id'] != -99]
        trajectory_dfs = [Trajectory(traj_id, 'Trajectory', df) for traj_id, df in trajectories_df.groupby('traj_id')]

        clusters_df = user_df[user_df['cluster'] != -99]
        cluster_dfs = [Cluster(cluster, 'Cluster', df) for cluster, df in clusters_df.groupby('cluster')]

        all_dfs = trajectory_dfs + cluster_dfs
        all_dfs_ordered = sorted(all_dfs, key=lambda tuple: tuple.df.iloc[0]['timestamp'])

        # trajectories_df['traj_id'] = trajectories_df['user_ID'].astype(str) + '_' + trajectories_df['traj_id'].astype(str)

        for tuple in all_dfs_ordered:
            if count >= processing_limit:
                break

            points_dataframe = tuple.df

            print(f'Processing {tuple.source} {tuple.id}')
            print(f'Starting point: ({points_dataframe.iloc[0]["lat"]}, {points_dataframe.iloc[0]["lng"]})')
            print(f'Ending point: ({points_dataframe.iloc[-1]["lat"]}, {points_dataframe.iloc[-1]["lng"]})')
            print(f'Number of points: {len(points_dataframe)}')

            distance_beginning_to_end = haversine(points_dataframe.iloc[0]['lat'],
                                                  points_dataframe.iloc[0]['lng'],
                                                  points_dataframe.iloc[-1]['lat'],
                                                  points_dataframe.iloc[-1]['lng'])
            print(f'Distance covered (first to last): {distance_beginning_to_end}')

            points_dataframe_minus_last = points_dataframe[:-1]
            bounds = (points_dataframe_minus_last['lng'].min(),
                      points_dataframe_minus_last['lat'].min(),
                      points_dataframe_minus_last['lng'].max(),
                      points_dataframe_minus_last['lat'].max())
            corner1 = (bounds[1], bounds[0])
            corner2 = (bounds[3], bounds[2])
            distance_across = haversine(corner1[0], corner1[1], corner2[0], corner2[1])
            print(f'Max distance covered (excluding last): {distance_across}')

            ## ADD TIME GAP TO DEMONSTRATE WHETHER LONG GAPS CREATES STOPS OF ONE DATAPOINT WHENEVER TIME + DISTANCE ARE LARGE ENOUGH

            if len(points_dataframe) < 2:
                longest_time_gap_minutes = 0.0
                final_time_gap = 0.0
            else:
                longest_time_gap_minutes = (points_dataframe['timestamp'] - points_dataframe['timestamp'].shift(1)).max().total_seconds() / 60
                final_time_gap = (points_dataframe.iloc[-1]['timestamp'] - points_dataframe.iloc[-2]['timestamp']).total_seconds() / 60
            print(f'Longest time gap: {longest_time_gap_minutes}')
            print(f'Time gap before end: {final_time_gap}')

            plot_trip_on_map(points_dataframe, include_transit=False)

            count += 1

    except Exception as e:
        print(f"Unexpected error processing user: {user}")
        print(f"Error: {e}")

    user_count += 1
    print(f'Appended {count} rows')
    total_trips_count += count

print("Done!")
print(f"Total trips processed: {total_trips_count}")
print(f"Trips filtered: {filtered_trips_count}")

In [ ]:
# Generate Spectus database metrics for 2000 users
from datetime import date
import pandas as pd
from pathlib import Path
from mode_analysis_main.src.calculate_speed import haversine
from mode_analysis_main.src.plot import plot_trip_on_map

from collections import namedtuple
Trajectory = namedtuple('Trajectory', ['id', 'source', 'df'])
Cluster = namedtuple('Cluster', ['id', 'source', 'df'])

today = date.today().strftime("%Y%m%d")

input_path = f'./outputs/seattle_2000_smaller_15min_500m_no_clustering.csv'

input_df = pd.read_csv(input_path, low_memory=False)
input_df = input_df.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
input_df['timestamp'] = pd.to_datetime(input_df['timestamp'])

processing_limit = 50
user_count = 0
total_trips_count = 0
filtered_trips_count = 0
for user, user_df in input_df.groupby('user_ID'):

    if total_trips_count >= processing_limit:
        break

    count = 0
    try:
        print(f'Processing user {user_count}')

        # Remove stop points
        trajectories_df = user_df[user_df['traj_id'] != -99]
        trajectory_dfs = [Trajectory(traj_id, 'Trajectory', df) for traj_id, df in trajectories_df.groupby('traj_id')]

        clusters_df = user_df[user_df['cluster'] != -99]
        cluster_dfs = [Cluster(cluster, 'Cluster', df) for cluster, df in clusters_df.groupby('cluster')]

        all_dfs = trajectory_dfs + cluster_dfs
        all_dfs_ordered = sorted(all_dfs, key=lambda tuple: tuple.df.iloc[0]['timestamp'])

        # trajectories_df['traj_id'] = trajectories_df['user_ID'].astype(str) + '_' + trajectories_df['traj_id'].astype(str)

        for tuple in all_dfs_ordered:
            if count >= processing_limit:
                break

            points_dataframe = tuple.df

            print(f'Processing {tuple.source} {tuple.id}')
            print(f'Starting point: ({points_dataframe.iloc[0]["lat"]}, {points_dataframe.iloc[0]["lng"]})')
            print(f'Ending point: ({points_dataframe.iloc[-1]["lat"]}, {points_dataframe.iloc[-1]["lng"]})')
            print(f'Number of points: {len(points_dataframe)}')

            distance_beginning_to_end = haversine(points_dataframe.iloc[0]['lat'],
                                                  points_dataframe.iloc[0]['lng'],
                                                  points_dataframe.iloc[-1]['lat'],
                                                  points_dataframe.iloc[-1]['lng'])
            print(f'Distance covered (first to last): {distance_beginning_to_end}')

            points_dataframe_minus_last = points_dataframe[:-1]
            bounds = (points_dataframe_minus_last['lng'].min(),
                      points_dataframe_minus_last['lat'].min(),
                      points_dataframe_minus_last['lng'].max(),
                      points_dataframe_minus_last['lat'].max())
            corner1 = (bounds[1], bounds[0])
            corner2 = (bounds[3], bounds[2])
            distance_across = haversine(corner1[0], corner1[1], corner2[0], corner2[1])
            print(f'Max distance covered (excluding last): {distance_across}')

            ## ADD TIME GAP TO DEMONSTRATE WHETHER LONG GAPS CREATES STOPS OF ONE DATAPOINT WHENEVER TIME + DISTANCE ARE LARGE ENOUGH

            if len(points_dataframe) < 2:
                longest_time_gap_minutes = 0.0
                final_time_gap = 0.0
            else:
                longest_time_gap_minutes = (points_dataframe['timestamp'] - points_dataframe['timestamp'].shift(1)).max().total_seconds() / 60
                final_time_gap = (points_dataframe.iloc[-1]['timestamp'] - points_dataframe.iloc[-2]['timestamp']).total_seconds() / 60
            print(f'Longest time gap: {longest_time_gap_minutes}')
            print(f'Time gap before end: {final_time_gap}')

            plot_trip_on_map(points_dataframe, include_transit=False)

            count += 1

    except Exception as e:
        print(f"Unexpected error processing user: {user}")
        print(f"Error: {e}")

    user_count += 1
    print(f'Appended {count} rows')
    total_trips_count += count

print("Done!")
print(f"Total trips processed: {total_trips_count}")
print(f"Trips filtered: {filtered_trips_count}")

In [ ]:
# Generate Spectus database metrics for 2000 users
from datetime import date
import pandas as pd
from pathlib import Path
from mode_analysis_main.src.calculate_speed import haversine
from mode_analysis_main.src.plot import plot_trip_on_map

from collections import namedtuple
Trajectory = namedtuple('Trajectory', ['id', 'source', 'df'])
Cluster = namedtuple('Cluster', ['id', 'source', 'df'])

today = date.today().strftime("%Y%m%d")

input_path = f'./outputs/seattle_2000_smaller_20min_200m_no_clustering.csv'

input_df = pd.read_csv(input_path, low_memory=False)
input_df = input_df.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
input_df['timestamp'] = pd.to_datetime(input_df['timestamp'])

processing_limit = 50
user_count = 0
total_trips_count = 0
filtered_trips_count = 0
for user, user_df in input_df.groupby('user_ID'):

    if total_trips_count >= processing_limit:
        break

    count = 0
    try:
        print(f'Processing user {user_count}')

        # Remove stop points
        trajectories_df = user_df[user_df['traj_id'] != -99]
        trajectory_dfs = [Trajectory(traj_id, 'Trajectory', df) for traj_id, df in trajectories_df.groupby('traj_id')]

        clusters_df = user_df[user_df['cluster'] != -99]
        cluster_dfs = [Cluster(cluster, 'Cluster', df) for cluster, df in clusters_df.groupby('cluster')]

        all_dfs = trajectory_dfs + cluster_dfs
        all_dfs_ordered = sorted(all_dfs, key=lambda tuple: tuple.df.iloc[0]['timestamp'])

        # trajectories_df['traj_id'] = trajectories_df['user_ID'].astype(str) + '_' + trajectories_df['traj_id'].astype(str)

        for tuple in all_dfs_ordered:
            if count >= processing_limit:
                break

            points_dataframe = tuple.df

            print(f'Processing {tuple.source} {tuple.id}')
            print(f'Starting point: ({points_dataframe.iloc[0]["lat"]}, {points_dataframe.iloc[0]["lng"]})')
            print(f'Ending point: ({points_dataframe.iloc[-1]["lat"]}, {points_dataframe.iloc[-1]["lng"]})')
            print(f'Number of points: {len(points_dataframe)}')

            distance_beginning_to_end = haversine(points_dataframe.iloc[0]['lat'],
                                                  points_dataframe.iloc[0]['lng'],
                                                  points_dataframe.iloc[-1]['lat'],
                                                  points_dataframe.iloc[-1]['lng'])
            print(f'Distance covered (first to last): {distance_beginning_to_end}')

            points_dataframe_minus_last = points_dataframe[:-1]
            bounds = (points_dataframe_minus_last['lng'].min(),
                      points_dataframe_minus_last['lat'].min(),
                      points_dataframe_minus_last['lng'].max(),
                      points_dataframe_minus_last['lat'].max())
            corner1 = (bounds[1], bounds[0])
            corner2 = (bounds[3], bounds[2])
            distance_across = haversine(corner1[0], corner1[1], corner2[0], corner2[1])
            print(f'Max distance covered (excluding last): {distance_across}')

            ## ADD TIME GAP TO DEMONSTRATE WHETHER LONG GAPS CREATES STOPS OF ONE DATAPOINT WHENEVER TIME + DISTANCE ARE LARGE ENOUGH

            if len(points_dataframe) < 2:
                longest_time_gap_minutes = 0.0
                final_time_gap = 0.0
            else:
                longest_time_gap_minutes = (points_dataframe['timestamp'] - points_dataframe['timestamp'].shift(1)).max().total_seconds() / 60
                final_time_gap = (points_dataframe.iloc[-1]['timestamp'] - points_dataframe.iloc[-2]['timestamp']).total_seconds() / 60
            print(f'Longest time gap: {longest_time_gap_minutes}')
            print(f'Time gap before end: {final_time_gap}')

            plot_trip_on_map(points_dataframe, include_transit=False)

            count += 1

    except Exception as e:
        print(f"Unexpected error processing user: {user}")
        print(f"Error: {e}")

    user_count += 1
    print(f'Appended {count} rows')
    total_trips_count += count

print("Done!")
print(f"Total trips processed: {total_trips_count}")
print(f"Trips filtered: {filtered_trips_count}")

In [ ]:
# Generate Spectus database metrics for 2000 users
from datetime import date
import pandas as pd
from pathlib import Path
from mode_analysis_main.src.calculate_speed import haversine
from mode_analysis_main.src.plot import plot_trip_on_map

from collections import namedtuple
Trajectory = namedtuple('Trajectory', ['id', 'source', 'df'])
Cluster = namedtuple('Cluster', ['id', 'source', 'df'])

today = date.today().strftime("%Y%m%d")

input_path = f'./outputs/seattle_2000_smaller_30min_200m_no_clustering.csv'

input_df = pd.read_csv(input_path, low_memory=False)
input_df = input_df.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
input_df['timestamp'] = pd.to_datetime(input_df['timestamp'])

processing_limit = 50
user_count = 0
total_trips_count = 0
filtered_trips_count = 0
for user, user_df in input_df.groupby('user_ID'):

    if total_trips_count >= processing_limit:
        break

    count = 0
    try:
        print(f'Processing user {user_count}')

        # Remove stop points
        trajectories_df = user_df[user_df['traj_id'] != -99]
        trajectory_dfs = [Trajectory(traj_id, 'Trajectory', df) for traj_id, df in trajectories_df.groupby('traj_id')]

        clusters_df = user_df[user_df['cluster'] != -99]
        cluster_dfs = [Cluster(cluster, 'Cluster', df) for cluster, df in clusters_df.groupby('cluster')]

        all_dfs = trajectory_dfs + cluster_dfs
        all_dfs_ordered = sorted(all_dfs, key=lambda tuple: tuple.df.iloc[0]['timestamp'])

        # trajectories_df['traj_id'] = trajectories_df['user_ID'].astype(str) + '_' + trajectories_df['traj_id'].astype(str)

        for tuple in all_dfs_ordered:
            if count >= processing_limit:
                break

            points_dataframe = tuple.df

            print(f'Processing {tuple.source} {tuple.id}')
            print(f'Starting point: ({points_dataframe.iloc[0]["lat"]}, {points_dataframe.iloc[0]["lng"]})')
            print(f'Ending point: ({points_dataframe.iloc[-1]["lat"]}, {points_dataframe.iloc[-1]["lng"]})')
            print(f'Number of points: {len(points_dataframe)}')

            distance_beginning_to_end = haversine(points_dataframe.iloc[0]['lat'],
                                                  points_dataframe.iloc[0]['lng'],
                                                  points_dataframe.iloc[-1]['lat'],
                                                  points_dataframe.iloc[-1]['lng'])
            print(f'Distance covered (first to last): {distance_beginning_to_end}')

            points_dataframe_minus_last = points_dataframe[:-1]
            bounds = (points_dataframe_minus_last['lng'].min(),
                      points_dataframe_minus_last['lat'].min(),
                      points_dataframe_minus_last['lng'].max(),
                      points_dataframe_minus_last['lat'].max())
            corner1 = (bounds[1], bounds[0])
            corner2 = (bounds[3], bounds[2])
            distance_across = haversine(corner1[0], corner1[1], corner2[0], corner2[1])
            print(f'Max distance covered (excluding last): {distance_across}')

            ## ADD TIME GAP TO DEMONSTRATE WHETHER LONG GAPS CREATES STOPS OF ONE DATAPOINT WHENEVER TIME + DISTANCE ARE LARGE ENOUGH

            if len(points_dataframe) < 2:
                longest_time_gap_minutes = 0.0
                final_time_gap = 0.0
            else:
                longest_time_gap_minutes = (points_dataframe['timestamp'] - points_dataframe['timestamp'].shift(1)).max().total_seconds() / 60
                final_time_gap = (points_dataframe.iloc[-1]['timestamp'] - points_dataframe.iloc[-2]['timestamp']).total_seconds() / 60
            print(f'Longest time gap: {longest_time_gap_minutes}')
            print(f'Time gap before end: {final_time_gap}')

            plot_trip_on_map(points_dataframe, include_transit=False)

            count += 1

    except Exception as e:
        print(f"Unexpected error processing user: {user}")
        print(f"Error: {e}")

    user_count += 1
    print(f'Appended {count} rows')
    total_trips_count += count

print("Done!")
print(f"Total trips processed: {total_trips_count}")
print(f"Trips filtered: {filtered_trips_count}")

In [ ]:
# Generate Spectus database metrics for 2000 users
from datetime import date
import pandas as pd
from pathlib import Path
from mode_analysis_main.src.calculate_speed import haversine
from mode_analysis_main.src.plot import plot_trip_on_map

from collections import namedtuple
Trajectory = namedtuple('Trajectory', ['id', 'source', 'df'])
Cluster = namedtuple('Cluster', ['id', 'source', 'df'])

today = date.today().strftime("%Y%m%d")

input_path = f'./outputs/seattle_2000_smaller_20min_200m_100mCluster.csv'

input_df = pd.read_csv(input_path, low_memory=False)
input_df = input_df.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
input_df['timestamp'] = pd.to_datetime(input_df['timestamp'])

processing_limit = 50
user_count = 0
total_trips_count = 0
filtered_trips_count = 0
for user, user_df in input_df.groupby('user_ID'):

    if total_trips_count >= processing_limit:
        break

    count = 0
    try:
        print(f'Processing user {user_count}')

        # Remove stop points
        trajectories_df = user_df[user_df['traj_id'] != -99]
        trajectory_dfs = [Trajectory(traj_id, 'Trajectory', df) for traj_id, df in trajectories_df.groupby('traj_id')]

        clusters_df = user_df[user_df['cluster'] != -99]
        cluster_dfs = [Cluster(cluster, 'Cluster', df) for cluster, df in clusters_df.groupby('cluster')]

        all_dfs = trajectory_dfs + cluster_dfs
        all_dfs_ordered = sorted(all_dfs, key=lambda tuple: tuple.df.iloc[0]['timestamp'])

        # trajectories_df['traj_id'] = trajectories_df['user_ID'].astype(str) + '_' + trajectories_df['traj_id'].astype(str)

        for tuple in all_dfs_ordered:
            if count >= processing_limit:
                break

            points_dataframe = tuple.df

            print(f'Processing {tuple.source} {tuple.id}')
            print(f'Starting point: ({points_dataframe.iloc[0]["lat"]}, {points_dataframe.iloc[0]["lng"]})')
            print(f'Ending point: ({points_dataframe.iloc[-1]["lat"]}, {points_dataframe.iloc[-1]["lng"]})')
            print(f'Number of points: {len(points_dataframe)}')

            distance_beginning_to_end = haversine(points_dataframe.iloc[0]['lat'],
                                                  points_dataframe.iloc[0]['lng'],
                                                  points_dataframe.iloc[-1]['lat'],
                                                  points_dataframe.iloc[-1]['lng'])
            print(f'Distance covered (first to last): {distance_beginning_to_end}')

            points_dataframe_minus_last = points_dataframe[:-1]
            bounds = (points_dataframe_minus_last['lng'].min(),
                      points_dataframe_minus_last['lat'].min(),
                      points_dataframe_minus_last['lng'].max(),
                      points_dataframe_minus_last['lat'].max())
            corner1 = (bounds[1], bounds[0])
            corner2 = (bounds[3], bounds[2])
            distance_across = haversine(corner1[0], corner1[1], corner2[0], corner2[1])
            print(f'Max distance covered (excluding last): {distance_across}')

            ## ADD TIME GAP TO DEMONSTRATE WHETHER LONG GAPS CREATES STOPS OF ONE DATAPOINT WHENEVER TIME + DISTANCE ARE LARGE ENOUGH

            if len(points_dataframe) < 2:
                longest_time_gap_minutes = 0.0
                final_time_gap = 0.0
            else:
                longest_time_gap_minutes = (points_dataframe['timestamp'] - points_dataframe['timestamp'].shift(1)).max().total_seconds() / 60
                final_time_gap = (points_dataframe.iloc[-1]['timestamp'] - points_dataframe.iloc[-2]['timestamp']).total_seconds() / 60
            print(f'Longest time gap: {longest_time_gap_minutes}')
            print(f'Time gap before end: {final_time_gap}')

            plot_trip_on_map(points_dataframe, include_transit=False)

            count += 1

    except Exception as e:
        print(f"Unexpected error processing user: {user}")
        print(f"Error: {e}")

    user_count += 1
    print(f'Appended {count} rows')
    total_trips_count += count

print("Done!")
print(f"Total trips processed: {total_trips_count}")
print(f"Trips filtered: {filtered_trips_count}")

In [ ]:
# Generate Spectus database metrics for 2000 users
from datetime import date
import pandas as pd
from pathlib import Path
from mode_analysis_main.src.calculate_speed import haversine
from mode_analysis_main.src.plot import plot_trip_on_map

from collections import namedtuple
Trajectory = namedtuple('Trajectory', ['id', 'source', 'df'])
Cluster = namedtuple('Cluster', ['id', 'source', 'df'])

today = date.today().strftime("%Y%m%d")

input_path = f'./outputs/seattle_2000_processed_15min_stop_100m_radius_small.csv'

input_df = pd.read_csv(input_path, low_memory=False)
input_df = input_df.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
input_df['timestamp'] = pd.to_datetime(input_df['timestamp'])

processing_limit = 75
user_count = 0
total_trips_count = 0
filtered_trips_count = 0
for user, user_df in input_df.groupby('user_ID'):

    if total_trips_count >= processing_limit:
        break

    count = 0
    try:
        print(f'Processing user {user_count}')

        # Remove stop points
        trajectories_df = user_df[user_df['traj_id'] != -99]
        trajectory_dfs = [Trajectory(traj_id, 'Trajectory', df) for traj_id, df in trajectories_df.groupby('traj_id')]

        clusters_df = user_df[user_df['cluster'] != -99]
        cluster_dfs = [Cluster(cluster, 'Cluster', df) for cluster, df in clusters_df.groupby('cluster')]

        all_dfs = trajectory_dfs + cluster_dfs
        all_dfs_ordered = sorted(all_dfs, key=lambda tuple: tuple.df.iloc[0]['timestamp'])

        # trajectories_df['traj_id'] = trajectories_df['user_ID'].astype(str) + '_' + trajectories_df['traj_id'].astype(str)

        for tuple in all_dfs_ordered:
            if count >= processing_limit:
                break

            points_dataframe = tuple.df

            print(f'Processing {tuple.source} {tuple.id}')
            print(f'Starting point: ({points_dataframe.iloc[0]["lat"]}, {points_dataframe.iloc[0]["lng"]})')
            print(f'Ending point: ({points_dataframe.iloc[-1]["lat"]}, {points_dataframe.iloc[-1]["lng"]})')
            print(f'Number of points: {len(points_dataframe)}')

            distance_beginning_to_end = haversine(points_dataframe.iloc[0]['lat'],
                                                  points_dataframe.iloc[0]['lng'],
                                                  points_dataframe.iloc[-1]['lat'],
                                                  points_dataframe.iloc[-1]['lng'])
            print(f'Distance covered (first to last): {distance_beginning_to_end}')

            points_dataframe_minus_last = points_dataframe[:-1]
            bounds = (points_dataframe_minus_last['lng'].min(),
                      points_dataframe_minus_last['lat'].min(),
                      points_dataframe_minus_last['lng'].max(),
                      points_dataframe_minus_last['lat'].max())
            corner1 = (bounds[1], bounds[0])
            corner2 = (bounds[3], bounds[2])
            distance_across = haversine(corner1[0], corner1[1], corner2[0], corner2[1])
            print(f'Max distance covered (excluding last): {distance_across}')

            ## ADD TIME GAP TO DEMONSTRATE WHETHER LONG GAPS CREATES STOPS OF ONE DATAPOINT WHENEVER TIME + DISTANCE ARE LARGE ENOUGH

            if len(points_dataframe) < 2:
                longest_time_gap_minutes = 0.0
                final_time_gap = 0.0
            else:
                longest_time_gap_minutes = (points_dataframe['timestamp'] - points_dataframe['timestamp'].shift(1)).max().total_seconds() / 60
                final_time_gap = (points_dataframe.iloc[-1]['timestamp'] - points_dataframe.iloc[-2]['timestamp']).total_seconds() / 60
            print(f'Longest time gap: {longest_time_gap_minutes}')
            print(f'Time gap before end: {final_time_gap}')

            plot_trip_on_map(points_dataframe, include_transit=False)

            count += 1

    except Exception as e:
        print(f"Unexpected error processing user: {user}")
        print(f"Error: {e}")

    user_count += 1
    print(f'Appended {count} rows')
    total_trips_count += count

print("Done!")
print(f"Total trips processed: {total_trips_count}")
print(f"Trips filtered: {filtered_trips_count}")